# m41 lab · Locating activation origins from the ECG

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You build a simple forward model from a pacing site to a 12-lead ECG, train population regressors that map ECG
features to an origin, fit patient-specific regressions from a few pacing sites, and run an active pace-mapping loop
with a Gaussian process and expected improvement. A last part shows what happens when the training hearts differ
from the test hearts.

| Part | What you do | Data |
| --- | --- | --- |
| A | Forward model: ellipsoidal ventricle, double layer, nine electrodes, twelve leads; check eq-m41-integral | synthetic |
| B | Population regression from twelve QRS integrals: linear, support-vector, random forest; split by patient | synthetic |
| C | Patient-specific regression from three leads and a few pacing sites (eq-m41-linear) | synthetic |
| D | Active pace-mapping: Gaussian process and expected improvement against random pacing | synthetic |
| E | Training hearts that differ from test hearts | synthetic |
| F | Optional: a small neural network in PyTorch | synthetic |

The model is an **approximate model**: a closed ellipsoidal surface activated from one site, a uniform double layer
switched on where the surface has activated, an infinite homogeneous medium. It ignores the torso boundary, wall
thickness, fibre direction and repolarization. Sizes, speeds, electrode positions and noise are illustrative. The
120-ms integration window follows Zhou et al. 2018. Each part ends with **checks**; the lab runs in about a
minute. No public data set with pacing-site coordinates could be verified for CI, so this lab is synthetic only.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import origins as og

E = np.array(list(og.ELECTRODES.values()))
R0 = og.rotation((0, 0, 1), 40) @ og.rotation((0, 1, 0), 45)       # long axis tilted (illustrative)
SPEED = 0.6                                                         # mm/ms, illustrative

def patient(rng, tilt=0.0):
    """One virtual patient: ellipsoid size, orientation, position and speed varied (all illustrative)."""
    axes = np.array([45.0, 25.0, 25.0]) * rng.uniform(0.85, 1.15, 3)
    R = og.rotation(rng.normal(size=3), rng.normal(0, 12)) @ og.rotation((1, 0, 0), tilt) @ R0
    v, t = og.lv_mesh(axes, R=R, centre=rng.normal(0, 12, 3))
    n, a = og.normals_areas(v, t)
    return {"v": v, "t": t, "Omega": og.solid_angles(v, n, a, E), "D": og.edge_distances(v, t),
            "speed": SPEED * rng.uniform(0.8, 1.2)}

## Part A · From a pacing site to twelve leads

Activation times are mesh distances from the pacing site divided by the conduction speed. Each surface element
switches on a double layer at its activation time, and an electrode sees the solid angle of the activated part.
Plot the twelve leads for two pacing sites, then check eq-m41-integral: the integral of each lead over the first
120 ms equals a solid-angle-weighted sum of $(T - \tau_j)_+$.

In [ ]:
rng = np.random.default_rng(41)
P = patient(rng)
t_ms = np.arange(0, 200, 0.5)
apex = int(np.argmax(P["v"] @ (R0 @ np.array([1.0, 0, 0]))))
base = int(np.argmin(P["v"] @ (R0 @ np.array([1.0, 0, 0]))))
fig, axs = plt.subplots(2, 6, figsize=(11, 4), sharex=True)
for site, color, name in [(apex, "tab:blue", "apex"), (base, "tab:orange", "base")]:
    ecg = og.leads12(og.electrode_potentials(P["Omega"], P["D"][site] / P["speed"], t_ms, rise=1.0))
    for ax, lead, sig in zip(axs.ravel(), og.LEADS, ecg):
        ax.plot(t_ms, sig, color=color, label=name)
        ax.set_title(lead, fontsize=9); ax.set_yticks([])
axs[0, 0].legend(fontsize=8); axs[1, 0].set_xlabel("time (ms)")
plt.tight_layout(); plt.show()

row_balance = np.abs(P["Omega"].sum(1)) / np.abs(P["Omega"]).sum(1)
tau = P["D"][apex] / P["speed"]
numeric = np.trapezoid(og.leads12(og.electrode_potentials(P["Omega"], tau, np.arange(0, 120.001, 0.05), rise=0.05)),
                       dx=0.05, axis=1)
closed = og.qrs_integral(P["Omega"], tau, 120.0)
print(f"solid angles of the closed surface cancel to {row_balance.max():.1e} of their absolute sum")
print("lead integrals, numeric vs eq-m41-integral (first four leads):", numeric[:4].round(3), closed[:4].round(3))

In [ ]:
# Checks
assert row_balance.max() < 1e-3
assert np.allclose(numeric, closed, rtol=0.02, atol=0.02 * np.abs(closed).max())

## Part B · A population model

Thirty virtual patients, 80 pacing sites each. Features: the twelve 120-ms QRS integrals with 10% noise
(illustrative). Target: the site's coordinates on a generic ventricle (the same vertex on a mean-shaped mesh).
Error: mesh distance on the generic surface between the predicted and the true site (eq-m41-geodesic). Train on
20 patients and test on the other 10; then repeat with a random split of pacing sites, as when sites from one
patient fall on both sides.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

gv, gt = og.lv_mesh(R=R0)
Dg = og.edge_distances(gv, gt)
X, site, pid = [], [], []
for p in range(30):
    Pp = patient(rng)
    for s in rng.choice(len(gv), 80, replace=False):
        X.append(og.qrs_integral(Pp["Omega"], Pp["D"][s] / Pp["speed"])); site.append(s); pid.append(p)
X, site, pid = np.array(X), np.array(site), np.array(pid)
X = X + rng.normal(0, 0.10 * X.std(0), X.shape)

def geo_error(pred_xyz, true_site):
    idx = np.argmin(((pred_xyz[:, None, :] - gv[None]) ** 2).sum(-1), axis=1)
    return Dg[idx, true_site]

models = {
    "linear (eq-m41-linear)": None,
    "support-vector": MultiOutputRegressor(make_pipeline(StandardScaler(), SVR(C=100, epsilon=1.0))),
    "random forest": RandomForestRegressor(300, random_state=0),
}
results_B = {}
for split_name, train in [("by patient", pid < 20), ("random sites", rng.random(len(pid)) < 2 / 3)]:
    test = ~train
    for name, m in models.items():
        if m is None:
            W, b = og.fit_linear(X[train], gv[site[train]])
            pred = X[test] @ W + b
        else:
            pred = m.fit(X[train], gv[site[train]]).predict(X[test])
        results_B[(split_name, name)] = geo_error(pred, site[test])
for (split_name, name), e in results_B.items():
    print(f"{split_name:13s} {name:24s} mean {e.mean():5.1f} mm, median {np.median(e):5.1f} mm")

In [ ]:
# Checks
lin = results_B[("by patient", "linear (eq-m41-linear)")]
assert 2 < lin.mean() < 20
assert results_B[("random sites", "linear (eq-m41-linear)")].mean() <= lin.mean() + 0.5

## Part C · A patient-specific model from a few pacing sites

With three leads (III, V2 and V6, the leads used by Zhou and colleagues for patient-specific localization) and
known coordinates of a few pacing sites in one patient, fit eq-m41-linear and predict an unknown origin. Four sites
is the minimum for three features plus an intercept. Compare sites chosen at random with sequential pacing: four
random sites, then each new site within 15 mm of the current prediction.

In [ ]:
LI = [og.LEADS.index(k) for k in ("III", "V2", "V6")]
seq, rnd = [], []
for trial in range(30):
    Pp = patient(rng)
    target = int(rng.integers(len(Pp["v"])))
    Xp = np.array([og.qrs_integral(Pp["Omega"], Pp["D"][s] / Pp["speed"])[LI] for s in range(len(Pp["v"]))])
    Xp = Xp + rng.normal(0, 0.05 * Xp.std(0), Xp.shape)
    preds = og.sequential_patient_fit(Xp, Pp["v"], Pp["D"], target, n_remote=4, n_total=12, radius=15.0, rng=rng)
    seq.append([Pp["D"][p, target] for p in preds])
    row = []
    for k in range(4, 13):
        s = rng.choice(len(Pp["v"]), k, replace=False)
        W, b = og.fit_linear(Xp[s], Pp["v"][s])
        idx = int(np.argmin(((Pp["v"] - (Xp[target] @ W + b)) ** 2).sum(1)))
        row.append(Pp["D"][idx, target])
    rnd.append(row)
seq, rnd = np.array(seq), np.array(rnd)
ks = np.arange(4, 13)
plt.plot(ks, rnd.mean(0), "o-", label="random sites")
plt.plot(ks, seq.mean(0), "s-", label="4 random, then near the prediction")
plt.xlabel("pacing sites with known coordinates"); plt.ylabel("mean error (mm)"); plt.legend(); plt.show()
print("mean error with 4, 8, 12 sites: random", rnd.mean(0)[[0, 4, 8]].round(1), " sequential", seq.mean(0)[[0, 4, 8]].round(1))

In [ ]:
# Checks
assert seq.mean(0)[-1] < seq.mean(0)[0]
assert seq.mean(0)[-1] <= rnd.mean(0)[-1]

## Part D · Active pace-mapping

The clinical target is an ECG (a premature beat or a tachycardia) whose origin is unknown. Pacing at a site gives
an ECG that can be compared with it (eq-m41-corr). A Gaussian process models that similarity over the surface
(eq-m41-gp), and expected improvement (eq-m41-ei) picks the next site, as in the approach of Meisenzahl et al.
2024. Start from two random sites, pace up to twelve, and compare with random pacing. The kernel settings (length
scale 20 mm, signal scale 0.3, noise 0.01) are illustrative and fixed.

In [ ]:
t_d = np.arange(0, 200, 2.0)
traces = {"expected improvement": [], "random": []}
for trial in range(12):
    Pp = patient(rng)
    target = int(rng.integers(len(Pp["v"])))
    ecgs = np.array([og.leads12(og.electrode_potentials(Pp["Omega"], Pp["D"][s] / Pp["speed"], t_d)) for s in range(len(Pp["v"]))])
    sim = np.array([og.pacemap_corr(ecgs[target], e) for e in ecgs])
    start = list(rng.choice(len(Pp["v"]), 2, replace=False))
    for name, rule in [("expected improvement", "ei"), ("random", "random")]:
        paced = og.active_pacemap(sim, Pp["v"], start, 12, ell=20.0, sf=0.3, sn=0.01, rule=rule,
                                  rng=np.random.default_rng(trial))
        traces[name].append([Pp["D"][paced[int(np.argmax(sim[paced[:m]]))], target] for m in range(2, 13)])
for name, tr in traces.items():
    tr = np.array(tr)
    plt.plot(range(2, 13), tr.mean(0), "o-", label=name)
    print(f"{name:21s} mean error of the best paced site after 4, 8, 12 pacings: {tr.mean(0)[[2, 6, 10]].round(1)} mm")
plt.xlabel("pacings"); plt.ylabel("distance of best paced site to origin (mm)"); plt.legend(); plt.show()

In [ ]:
# Checks
ei, rn = np.array(traces["expected improvement"]).mean(0), np.array(traces["random"]).mean(0)
assert ei[-1] < rn[-1] and ei[-1] < ei[0]

## Part E · When training hearts differ from test hearts

Train the linear population model on hearts with one orientation, then test it on hearts tilted by a further
25° (illustrative). Then train on a mix of both. Simulation-trained models meet the same problem when applied to
patients whose anatomy the simulations did not cover.

In [ ]:
def population(n_pat, tilt, n_sites=60):
    Xs, ss = [], []
    for _ in range(n_pat):
        Pp = patient(rng, tilt)
        for s in rng.choice(len(gv), n_sites, replace=False):
            Xs.append(og.qrs_integral(Pp["Omega"], Pp["D"][s] / Pp["speed"])); ss.append(s)
    Xs = np.array(Xs)
    return Xs + rng.normal(0, 0.10 * Xs.std(0), Xs.shape), np.array(ss)

Xa, sa = population(20, 0.0)
Xb, sb = population(20, 25.0)
Xb_test, sb_test = population(10, 25.0)
shift = {}
for name, (Xtr, str_) in {"trained on A": (Xa, sa), "trained on A + B": (np.r_[Xa, Xb], np.r_[sa, sb])}.items():
    W, b = og.fit_linear(Xtr, gv[str_])
    shift[name] = geo_error(Xb_test @ W + b, sb_test)
    print(f"{name:17s} tested on tilted hearts: mean error {shift[name].mean():.1f} mm")

In [ ]:
# Checks
assert shift["trained on A + B"].mean() < shift["trained on A"].mean()

## Part F · Optional: a small neural network

A two-layer perceptron from the twelve QRS integrals to coordinates, trained on the part B patients 1–20 and
tested on 21–30. Runs only if PyTorch is installed.

In [ ]:
try:
    import torch
except ImportError:
    torch = None
    if os.environ.get("EPML_REQUIRE_TORCH") == "1":
        raise
    print("PyTorch is not installed: part F skipped.")

if torch is not None:
    torch.manual_seed(0)
    tr, te = pid < 20, pid >= 20
    mu_x, sd_x = X[tr].mean(0), X[tr].std(0)
    Xt = torch.tensor((X[tr] - mu_x) / sd_x, dtype=torch.float32)
    Yt = torch.tensor(gv[site[tr]], dtype=torch.float32)
    net = torch.nn.Sequential(torch.nn.Linear(12, 64), torch.nn.ReLU(), torch.nn.Linear(64, 64), torch.nn.ReLU(), torch.nn.Linear(64, 3))
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    for epoch in range(800):
        opt.zero_grad()
        loss = ((net(Xt) - Yt) ** 2).mean()
        loss.backward()
        opt.step()
    with torch.no_grad():
        pred = net(torch.tensor((X[te] - mu_x) / sd_x, dtype=torch.float32)).numpy()
    mlp_err = geo_error(pred, site[te])
    print(f"neural network: mean error {mlp_err.mean():.1f} mm (linear model of part B: {lin.mean():.1f} mm)")

In [ ]:
# Checks
if torch is not None:
    assert mlp_err.mean() < 3 * lin.mean()

## Where to go next

- In part B, raise the feature noise to 30%. Which model degrades least?
- In part D, set the GP length scale to 5 mm and to 60 mm. What happens to the number of pacings needed?
- In part E, how much of the error comes from orientation, and how much from size? Vary one at a time.